# Fink/LSST — Dipole Concentration per diaObject (from cached parquet + API sources)

## Strategy

This notebook is the **read-from-cache** variant of `03_dipoleobjectcorr.ipynb`.

The cone-search step is **replaced** by reading the per-DDF parquet files already
produced by `01c_fink_dipoles_per_ddf.ipynb` and stored in `data_DIPOLES_01c/`.
This is exactly the same approach used by `02_fink_dipoles_uniformity.ipynb`.

Each parquet contains **one row per alert** for that DDF, with all crossmatch
columns (`f:xm_*`) and the key field `r:nDiaSources` (total detections for that
diaObject at the time of the alert).

### Pipeline

1. **Read parquets** from `data_DIPOLES_01c/` (one file per DDF, no API call).
2. **Deduplicate** by `r:diaObjectId`, keep the row with the maximum `r:nDiaSources`.
3. **Pre-select** objects with `r:nDiaSources >= NDIASOURCES_MIN` (e.g. 1000).
4. **Download full diaSources** for the pre-selected objects via `/api/v1/sources`,
   requesting all dipole columns **and** aperture-flux columns (`r:apFlux`, `r:apFluxErr`).
5. **Count dipoles** per object and per band from the downloaded diaSources.
6. **Lorenz curve + Gini coefficient** to quantify dipole concentration across objects.
7. **Stacked histograms** per object and per DDF.
8. **psfFlux − apFlux difference histograms** (dipole vs non-dipole) to check whether
   dipole-flagged sources show a characteristic aperture correction offset.
9. **Three-panel light curve** for the top high-dipole objects.
10. **Angular stability** of dipole direction: circular standard deviation per object.

## Data source

Parquet files: `data_DIPOLES_01c/{field}_alerts.parquet`  
**No cone-search API call is made here.**


- author : Sylvie Dagoret-Campagne
- affiliation : IJCLab/IN2P3/CNRS, Université Paris-Saclay
- creation : 2026-05-26
- last update : 2026-05-27

## 1. Imports & configuration

In [ ]:
import os
import time
import warnings

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from astropy.time import Time

warnings.filterwarnings("ignore")
print(f"pandas  : {pd.__version__}")
print(f"numpy   : {np.__version__}")

In [ ]:
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → %matplotlib widget")
except ImportError:
    %matplotlib inline
    print("ipympl not found → %matplotlib inline")

In [ ]:
# ── Input: parquet files from notebook 01c (same pattern as notebook 02) ──────
DIR_DATA_IN = "data_DIPOLES_01c"

# ── Fink API (only used for /api/v1/sources download in section 5) ────────────
FINK_API = "https://api.lsst.fink-portal.org"

# ── Output directories ────────────────────────────────────────────────────────
NB_TAG = "DIPOLES_03b"
DIR_DATA = f"data_{NB_TAG}"
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_DATA, exist_ok=True)
os.makedirs(DIR_FIGS, exist_ok=True)
print(f"Input data : {os.path.abspath(DIR_DATA_IN)}")
print(f"Output data: {os.path.abspath(DIR_DATA)}")
print(f"Figures    : {os.path.abspath(DIR_FIGS)}")

# ── DDF definitions (must match 01c) ─────────────────────────────────────────
DEEP_FIELDS = {
    "COSMOS": (150.1191, 2.2058),
    "ELAIS-S1": (9.4500, -44.000),
    "ECDFS": (53.1250, -27.800),
    "EDFS-a": (58.9000, -49.315),
    "EDFS-b": (63.6000, -47.600),
    "EDFS": (61.2400, -48.423),
    "M49": (187.4000, 8.000),
}

# ── Pre-selection threshold on nDiaSources ────────────────────────────────────
# Only objects with r:nDiaSources >= NDIASOURCES_MIN are kept.
# Recommended: 1000.  Reduce to 500 or 200 if too few objects survive.
NDIASOURCES_MIN = 500

# ── Light curve plotting ──────────────────────────────────────────────────────
TOP_N_OBJECTS = 10  # max objects in detailed light curve plots

# ── Zero-point for flux → magnitude conversion (AB system, nJy) ──────────────
FLUX0_NJY = 3.631e9  # 1 Jy = 1e9 nJy;  m_AB = -2.5*log10(F/F0) with F0=3631 Jy

# ── Plotting style ────────────────────────────────────────────────────────────
BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}
BAND_ORDER = list("ugrizy")

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)


def savefig(name: str) -> None:
    """Save the current figure to PDF and PNG in DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  -> saved {name}.{{pdf,png}}")


print(f"Configuration done.  NDIASOURCES_MIN={NDIASOURCES_MIN}")

## 2. Load parquet files from `data_DIPOLES_01c`

Same loading pattern as `02_fink_dipoles_uniformity.ipynb`.
Each parquet file contains one row per alert for that DDF.  
**No API call is made here.**

Key columns used:
- `r:diaObjectId` — object identifier
- `r:nDiaSources` — total detections accumulated for this object (most recent alert)
- `r:ra`, `r:dec` — sky position
- `f:xm_gaiadr3_DR3Name`, `f:xm_simbad_otype`, … — crossmatch identifiers
- `f:main_label_crossmatch` — Fink classification label

In [ ]:
XM_COLS = [
    "f:xm_gaiadr3_DR3Name",
    "f:xm_gaiadr3_VarFlag",
    "f:xm_gaiadr3_Plx",
    "f:xm_gaiadr3_e_Plx",
    "f:xm_simbad_otype",
    "f:xm_vsx_Type",
    "f:xm_tns_fullname",
    "f:xm_legacydr8_pstar",
    "f:main_label_crossmatch",
    "f:main_label_classifier",
]
NULL_VALS = {"", "None", "nan", "Fail", "null", "NaN"}

ddf_alerts: dict[str, pd.DataFrame] = {}

for field_name in DEEP_FIELDS:
    pq = os.path.join(DIR_DATA_IN, f"{field_name}_alerts.parquet")
    if not os.path.exists(pq):
        print(f"[{field_name:12s}] parquet not found — skipping.")
        ddf_alerts[field_name] = pd.DataFrame()
        continue
    df = pd.read_parquet(pq)
    for col in ("r:ra", "r:dec", "r:nDiaSources"):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    if "r:isDipole" in df.columns:
        df["r:isDipole"] = (
            df["r:isDipole"]
            .map(
                lambda v: (
                    True
                    if str(v).strip().lower() in ("true", "1", "yes")
                    else False
                    if str(v).strip().lower() in ("false", "0", "no")
                    else pd.NA
                )
            )
            .astype("boolean")
        )
    ddf_alerts[field_name] = df
    n_tot = len(df)
    n_dip = int(df["r:isDipole"].fillna(False).sum()) if "r:isDipole" in df.columns else 0
    n_obj = df["r:diaObjectId"].nunique() if "r:diaObjectId" in df.columns else 0
    print(f"[{field_name:12s}] {n_tot:6d} alerts  |  {n_obj:5d} objects  |  {n_dip:5d} dipole alerts")

print("\nLoad complete.")

## 3. API wrapper and utilities

In [ ]:
def _post_json(url: str, payload: dict, timeout: int = 90):
    r = requests.post(url, json=payload, timeout=timeout)
    r.raise_for_status()
    return r.json()


def fetch_sources(diaObjectId, columns: str | None = None) -> pd.DataFrame:
    """Fetch ALL diaSources for one diaObjectId via /api/v1/sources."""
    payload = {"diaObjectId": str(diaObjectId), "output-format": "json"}
    if columns:
        payload["columns"] = columns
    raw = _post_json(f"{FINK_API}/api/v1/sources", payload)
    return pd.DataFrame(raw) if raw else pd.DataFrame()


def parse_dipole_bool(series: pd.Series) -> pd.Series:
    """Convert a dipole boolean column (bool / int / str) to a proper bool Series."""

    def _to_bool(v):
        if isinstance(v, bool):
            return v
        if isinstance(v, (int, float)):
            return bool(v)
        if isinstance(v, str):
            return v.strip().lower() in ("true", "1", "yes")
        return False

    return series.apply(_to_bool)


def mjd_to_datestr(mjd_array) -> list:
    """Convert MJD (TAI) → list of 'YYYY-MM-DD' strings."""
    t = Time(np.asarray(mjd_array, dtype=float), format="mjd", scale="tai")
    return [tt.strftime("%Y-%m-%d") for tt in t]


def add_date_axis_on_top(ax, mjd_values: np.ndarray, n_ticks: int = 8) -> None:
    """Add a secondary x-axis on top of *ax* showing calendar dates (YYYY-MM-DD)."""
    finite = mjd_values[np.isfinite(mjd_values)]
    if len(finite) < 2:
        return
    mjd_lo, mjd_hi = float(finite.min()), float(finite.max())
    if mjd_hi <= mjd_lo:
        return
    tick_mjd = np.linspace(mjd_lo, mjd_hi, max(3, min(n_ticks, len(finite))))
    tick_lbls = mjd_to_datestr(tick_mjd)
    ax_top = ax.twiny()
    ax_top.set_xlim(ax.get_xlim())
    ax_top.set_xticks(tick_mjd)
    ax_top.set_xticklabels(tick_lbls, rotation=40, ha="left", fontsize=7)
    ax_top.tick_params(axis="x", length=4, pad=2)
    ax_top.set_xlabel("Date (UTC)", fontsize=7, labelpad=6)


def flux_to_mag(flux_njy: np.ndarray) -> np.ndarray:
    """Convert flux in nJy to AB magnitude.  Non-positive values → NaN."""
    with np.errstate(invalid="ignore", divide="ignore"):
        mag = np.where(flux_njy > 0, -2.5 * np.log10(flux_njy / FLUX0_NJY), np.nan)
    return mag


print("API wrapper and utilities defined.")

## 4. Build the pre-selection catalogue from the parquets

**Deduplication strategy**: group by `r:diaObjectId` and keep:
- `r:nDiaSources` = **max** across all alerts (most recent = highest count)
- `r:ra`, `r:dec` = first occurrence
- `f:xm_*` = **mode of non-null values** (same as notebook `01`)

Then apply `r:nDiaSources >= NDIASOURCES_MIN`.

In [ ]:
def _mode_non_null(series: pd.Series):
    """Mode of a Series after dropping null-like string values."""
    vals = series.dropna().astype(str)
    vals = vals[~vals.isin(NULL_VALS)]
    return vals.mode().iloc[0] if not vals.empty else None


presel: dict[str, dict] = {}

for field_name, df in ddf_alerts.items():
    if df.empty or "r:diaObjectId" not in df.columns or "r:nDiaSources" not in df.columns:
        continue
    agg_rows = []
    for oid, grp in df.groupby("r:diaObjectId"):
        row = {
            "r:diaObjectId": oid,
            "r:nDiaSources": int(grp["r:nDiaSources"].max()),
            "r:ra": float(grp["r:ra"].iloc[0]) if "r:ra" in grp.columns else np.nan,
            "r:dec": float(grp["r:dec"].iloc[0]) if "r:dec" in grp.columns else np.nan,
        }
        for col in XM_COLS:
            row[col] = _mode_non_null(grp[col]) if col in grp.columns else None
        agg_rows.append(row)
    df_obj = pd.DataFrame(agg_rows)
    df_ok = df_obj[df_obj["r:nDiaSources"] >= NDIASOURCES_MIN]
    print(
        f"[{field_name:12s}]  {len(df_obj):5d} objects  →  "
        f"{len(df_ok):4d} with nDiaSources>={NDIASOURCES_MIN}"
    )
    for _, row in df_ok.iterrows():
        oid = str(row["r:diaObjectId"])
        if oid not in presel:
            presel[oid] = {
                "nDiaSources": int(row["r:nDiaSources"]),
                "ra": row["r:ra"],
                "dec": row["r:dec"],
                "field": field_name,
                "gaia_name": row.get("f:xm_gaiadr3_DR3Name"),
                "gaia_var": row.get("f:xm_gaiadr3_VarFlag"),
                "gaia_plx": row.get("f:xm_gaiadr3_Plx"),
                "simbad": row.get("f:xm_simbad_otype"),
                "vsx": row.get("f:xm_vsx_Type"),
                "tns": row.get("f:xm_tns_fullname"),
                "label": row.get("f:main_label_crossmatch"),
                "label_clf": row.get("f:main_label_classifier"),
                "pstar": row.get("f:xm_legacydr8_pstar"),
            }

print(f"\n=== Total pre-selected objects (nDiaSources>={NDIASOURCES_MIN}): {len(presel)} ===")

In [ ]:
df_presel = (
    pd.DataFrame([{"diaObjectId": oid, **meta} for oid, meta in presel.items()])
    .sort_values("nDiaSources", ascending=False)
    .reset_index(drop=True)
)
print(f"Pre-selection catalogue: {len(df_presel)} objects")
display(df_presel.head(20))

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
nv = df_presel["nDiaSources"].values
axes[0].hist(nv, bins=40, color="steelblue", edgecolor="white", lw=0.3)
axes[0].axvline(NDIASOURCES_MIN, color="tomato", lw=1.5, ls="--", label=f"threshold = {NDIASOURCES_MIN}")
axes[0].set_xlabel("nDiaSources (from parquet)")
axes[0].set_ylabel("N objects")
axes[0].set_title(f"nDiaSources distribution (all >= {NDIASOURCES_MIN})")
axes[0].legend(fontsize=8)
bins_log = np.logspace(np.log10(max(nv.min(), 1)), np.log10(nv.max() + 1), 40)
axes[1].hist(nv, bins=bins_log, color="steelblue", edgecolor="white", lw=0.3)
axes[1].axvline(NDIASOURCES_MIN, color="tomato", lw=1.5, ls="--")
axes[1].set_xscale("log")
axes[1].set_xlabel("nDiaSources")
axes[1].set_title("nDiaSources distribution (log)")
plt.tight_layout()
savefig(f"nDiaSources_distribution_min{NDIASOURCES_MIN}")
plt.show()

df_presel.to_parquet(os.path.join(DIR_DATA, "presel_catalogue.parquet"), index=False)
df_presel.to_csv(os.path.join(DIR_DATA, "presel_catalogue.csv"), index=False)
print("Pre-selection catalogue saved.")

## 5. Download full diaSources for all pre-selected objects

This is the only section that calls the Fink API.  All dipole columns are
requested explicitly, **plus the aperture-flux columns** `r:apFlux` and
`r:apFluxErr` needed for the psfFlux − apFlux diagnostic (section 8b).

Objects are processed in decreasing order of `nDiaSources`.

In [ ]:
COLS_SRC = (
    "r:diaObjectId,r:diaSourceId,r:midpointMjdTai,"
    "r:psfFlux,r:psfFluxErr,"
    "r:apFlux,r:apFluxErr,"
    "r:scienceFlux,r:scienceFluxErr,"
    "r:templateFlux,r:templateFluxErr,"
    "r:band,r:ra,r:dec,r:snr,"
    "r:visit,r:detector,r:x,r:y,"
    "r:isDipole,r:isNegative,r:dipoleFitAttempted,"
    "r:dipoleFluxDiff,r:dipoleFluxDiffErr,"
    "r:dipoleMeanFlux,r:dipoleMeanFluxErr,"
    "r:dipoleLength,r:dipoleAngle,"
    "r:dipoleNdata,r:dipoleChi2"
)

AP_FLUX_COLS = ("r:apFlux", "r:apFluxErr")


def _cast_src(df: pd.DataFrame) -> pd.DataFrame:
    """Cast columns to appropriate types after API download."""
    for col in ("r:isDipole", "r:isNegative", "r:dipoleFitAttempted"):
        if col in df.columns:
            df[col] = parse_dipole_bool(df[col].fillna(False))
    float_cols = (
        "r:midpointMjdTai",
        "r:psfFlux",
        "r:psfFluxErr",
        "r:apFlux",
        "r:apFluxErr",
        "r:scienceFlux",
        "r:scienceFluxErr",
        "r:templateFlux",
        "r:templateFluxErr",
        "r:snr",
        "r:dipoleFluxDiff",
        "r:dipoleFluxDiffErr",
        "r:dipoleMeanFlux",
        "r:dipoleMeanFluxErr",
        "r:dipoleLength",
        "r:dipoleAngle",
        "r:dipoleNdata",
        "r:dipoleChi2",
        "r:x",
        "r:y",
    )
    for col in float_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    for col in ("r:visit", "r:detector"):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
    return df


oids_sorted = df_presel["diaObjectId"].tolist()
src_cache: dict[str, pd.DataFrame] = {}

print(f"Downloading diaSources for {len(oids_sorted)} objects…")
for i, oid in enumerate(oids_sorted):
    try:
        df = fetch_sources(oid, columns=COLS_SRC)
        df = _cast_src(df)
        src_cache[oid] = df
        n_dip = int(df["r:isDipole"].sum()) if ("r:isDipole" in df.columns and not df.empty) else 0
        has_ap = "r:apFlux" in df.columns and df["r:apFlux"].notna().any()
        print(
            f"  [{i + 1:3d}/{len(oids_sorted)}] {oid}  "
            f"n_src={len(df):5d}  n_dip={n_dip:4d}  "
            f"apFlux={'yes' if has_ap else 'no':3s}  "
            f"field={presel[oid]['field']}"
        )
    except Exception as e:
        print(f"  [{i + 1:3d}/{len(oids_sorted)}] {oid}  ERROR: {e}")
        src_cache[oid] = pd.DataFrame()
    time.sleep(0.2)

n_ok = sum(1 for df in src_cache.values() if not df.empty)
print(f"\nDownload complete: {n_ok}/{len(oids_sorted)} objects with data.")

## 6. Compute per-object dipole statistics from diaSources

In [ ]:
stat_rows = []
for oid, df in src_cache.items():
    if df.empty or "r:isDipole" not in df.columns:
        continue
    meta = presel[oid]
    n_src = len(df)
    n_dip = int(df["r:isDipole"].sum())
    row = {
        "diaObjectId": oid,
        "field": meta["field"],
        "nDiaSources": meta["nDiaSources"],
        "n_src": n_src,
        "n_dipoles": n_dip,
        "dipole_fraction": n_dip / n_src if n_src > 0 else np.nan,
        "ra": meta["ra"],
        "dec": meta["dec"],
        "gaia_name": meta.get("gaia_name"),
        "simbad": meta.get("simbad"),
        "label": meta.get("label"),
        "label_clf": meta.get("label_clf"),
    }
    if "r:band" in df.columns:
        band_dip = df[df["r:isDipole"]].groupby("r:band").size().reindex(BAND_ORDER, fill_value=0)
        for b in BAND_ORDER:
            row[f"n_dip_{b}"] = int(band_dip.get(b, 0))
    stat_rows.append(row)

df_stats = pd.DataFrame(stat_rows).sort_values("n_dipoles", ascending=False).reset_index(drop=True)
print(f"Statistics computed for {len(df_stats)} objects.")
print(f"  with >= 1 dipole  : {(df_stats['n_dipoles'] >= 1).sum()}")
print(f"  with >= 5 dipoles : {(df_stats['n_dipoles'] >= 5).sum()}")
print(f"  with >= 20 dipoles: {(df_stats['n_dipoles'] >= 20).sum()}")
display(df_stats.head(20))

df_stats.to_parquet(os.path.join(DIR_DATA, "dipole_stats_from_sources.parquet"), index=False)
df_stats.to_csv(os.path.join(DIR_DATA, "dipole_stats_from_sources.csv"), index=False)
print("Saved dipole statistics.")

## 7. Stacked histogram: dipole count per object, stacked by band

Each bar = one `diaObjectId` (pre-selected, `nDiaSources >= NDIASOURCES_MIN`).  
The stacked colours show per-band dipole counts.  
Objects are sorted by total dipole count descending.

### The Lorenz curve and the Gini coefficient

The **Lorenz curve** (Corrado Lorenz, 1905), originally an economics tool to measure
income inequality, is reused here to quantify the **concentration of dipoles across
diaObjects**.

#### Construction

Objects are sorted from the least dipolaire to the most dipolaire.  
The curve plots:

- **x-axis**: cumulative fraction of objects (0 → 100 %)
- **y-axis**: cumulative fraction of the total dipole budget carried by those objects (0 → 100 %)

#### The two extreme cases

| Curve shape | Meaning |
|-------------|----------|
| **Diagonal** (dashed grey) | Perfectly uniform: every object has the same dipole rate. 10 % of objects → 10 % of dipoles. |
| **Strongly bowed** (far below diagonal) | Highly concentrated: a tiny minority of objects carries almost all dipoles. |

#### Physical interpretation in this context

The curve answers:
> *Are dipoles uniformly spread across all well-observed diaObjects, or do a few objects monopolise them?*

A strongly bowed curve — large *X* in the annotation *"top 10% → X% of dipoles"* —
implies that dipoles are not a diffuse field-wide artefact but are concentrated on
specific objects: bright stars with poorly subtracted PSF wings, objects on bad CCD
columns, variable objects whose template has become outdated, etc.

#### The Gini coefficient

The **Gini coefficient** $G$ summarises the whole curve in a single scalar.  
It equals **twice the area between the diagonal and the Lorenz curve**:

$$G = 1 - 2\int_0^1 L(x)\,dx \approx 1 - 2\,\mathrm{trapz}\bigl(L(x),\, x\bigr)$$

| Value | Meaning |
|-------|---------|
| $G = 0$ | Perfectly uniform (curve = diagonal) |
| $G = 1$ | Perfect concentration (one object carries all dipoles) |
| $G > 0.6$ | Very unequal — dipoles driven by a small number of pathological objects |

The coefficient is computed numerically with the trapezoidal rule on the sorted
cumulative arrays and printed directly below the Lorenz curve plot.

In [ ]:
df_dip_nz = df_stats[df_stats["n_dipoles"] > 0].copy()

if df_dip_nz.empty:
    print("No dipoles found in the pre-selected objects.")
else:
    N_SHOW = min(80, len(df_dip_nz))
    top_df = df_dip_nz.head(N_SHOW)
    x_pos = np.arange(N_SHOW)
    bottom = np.zeros(N_SHOW)

    fig, ax = plt.subplots(figsize=(max(12, N_SHOW * 0.22), 5))
    for band in BAND_ORDER:
        col = f"n_dip_{band}"
        if col not in top_df.columns:
            continue
        vals = top_df[col].values.astype(float)
        ax.bar(
            x_pos,
            vals,
            bottom=bottom,
            color=BAND_COLORS[band],
            edgecolor="white",
            lw=0.3,
            label=f"band {band}",
            width=0.85,
        )
        bottom += vals
    ax.set_xticks(x_pos)
    ax.set_xticklabels([str(oid)[-8:] for oid in top_df["diaObjectId"]], rotation=90, fontsize=6)
    ax.set_xlabel("diaObjectId (last 8 digits)")
    ax.set_ylabel("Number of dipole detections")
    ax.set_title(
        f"Dipole count per diaObject — top {N_SHOW} (stacked by band)\n"
        f"Pre-selection: nDiaSources >= {NDIASOURCES_MIN}  |  "
        f"Objects with >=1 dipole: {len(df_dip_nz)}"
    )
    ax.legend(loc="upper right", fontsize=8, ncol=3)
    plt.tight_layout()
    savefig(f"dipole_stacked_per_object_min{NDIASOURCES_MIN}")
    plt.show()

In [ ]:
# ── Dipole count distribution + Lorenz curve + Gini coefficient ───────────────
if not df_dip_nz.empty:
    counts = df_dip_nz["n_dipoles"].values
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    # --- Linear histogram ---
    axes[0].hist(counts, bins=40, color="steelblue", edgecolor="white", lw=0.3)
    axes[0].set_xlabel("n_dipoles per object")
    axes[0].set_ylabel("N objects")
    axes[0].set_title(f"Dipole count distribution (linear)  [nDiaSrc>={NDIASOURCES_MIN}]")

    # --- Log-log histogram ---
    bins_log = np.logspace(0, np.log10(counts.max() + 1), 30)
    axes[1].hist(counts, bins=bins_log, color="tomato", edgecolor="white", lw=0.3)
    axes[1].set_xscale("log")
    axes[1].set_yscale("log")
    axes[1].set_xlabel("n_dipoles per object")
    axes[1].set_title("Dipole count distribution (log-log)")

    # --- Lorenz curve ---
    sc = np.sort(counts)  # ascending — poorest first
    cum = np.cumsum(sc) / sc.sum()  # cumulative fraction of dipoles
    obj = np.arange(1, len(sc) + 1) / len(sc)  # cumulative fraction of objects

    axes[2].plot(obj * 100, cum * 100, color="steelblue", lw=2, label="Lorenz curve")
    axes[2].fill_between(obj * 100, obj * 100, cum * 100, alpha=0.15, color="steelblue", label="Gini area")
    axes[2].plot([0, 100], [0, 100], "--", color="grey", lw=1, label="equal distribution")

    # Annotate top 10 %
    idx10 = max(1, int(0.10 * len(sc)))
    # top 10% → the 10% with most dipoles = last 10% of the sorted array
    frac10 = (1 - cum[-(idx10)]) * 100  # fraction carried by top 10%
    axes[2].axvline(90, color="tomato", lw=1, ls=":")
    axes[2].text(
        74, frac10 / 2 + 5, f"top 10% objects\n→ {frac10:.0f}% of dipoles", color="tomato", fontsize=8
    )

    # ── Gini coefficient (trapezoidal rule) ──────────────────────────────────
    # G = 1 - 2 * integral of L(x) dx  (x and L both in [0,1])
    gini = 1.0 - 2.0 * float(np.trapz(cum, obj))
    axes[2].text(
        5,
        88,
        f"Gini = {gini:.3f}",
        fontsize=10,
        color="steelblue",
        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="steelblue", alpha=0.8),
    )

    axes[2].set_xlabel("Cumulative fraction of objects (%, poorest first)")
    axes[2].set_ylabel("Cumulative fraction of dipoles (%)")
    axes[2].set_title("Lorenz curve — dipole concentration")
    axes[2].set_xlim(0, 100)
    axes[2].set_ylim(0, 100)
    axes[2].legend(fontsize=8, loc="upper left")

    plt.tight_layout()
    savefig(f"dipole_distribution_lorenz_min{NDIASOURCES_MIN}")
    plt.show()

    print(f"\nGini coefficient = {gini:.4f}")
    if gini > 0.6:
        print("  → G > 0.6: very unequal — dipoles dominated by a small number of objects.")
    elif gini > 0.3:
        print("  → 0.3 < G ≤ 0.6: moderately unequal distribution.")
    else:
        print("  → G ≤ 0.3: relatively uniform distribution.")

## 8. Select top high-dipole objects for light curve inspection

In [ ]:
top_sel = df_stats[df_stats["n_dipoles"] > 0].head(TOP_N_OBJECTS).copy()
print(f"Top {TOP_N_OBJECTS} objects by dipole count:")
display(
    top_sel[
        [
            "diaObjectId",
            "field",
            "nDiaSources",
            "n_src",
            "n_dipoles",
            "dipole_fraction",
            "gaia_name",
            "simbad",
            "label",
        ]
    ]
)

fig, ax = plt.subplots(figsize=(7, 5))
for fld in DEEP_FIELDS:
    sub = df_stats[df_stats["field"] == fld]
    ax.scatter(sub["n_src"], sub["n_dipoles"], s=12, alpha=0.5, label=fld)
ax.scatter(
    top_sel["n_src"].values,
    top_sel["n_dipoles"].values,
    s=90,
    marker="*",
    color="gold",
    edgecolors="k",
    lw=0.5,
    zorder=5,
    label=f"top {TOP_N_OBJECTS}",
)
ax.set_xlabel("n_src (diaSources downloaded)")
ax.set_ylabel("n_dipoles")
ax.set_title(f"Sources vs dipoles per diaObject  [nDiaSrc>={NDIASOURCES_MIN}]")
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
savefig(f"nsrc_vs_ndipoles_scatter_min{NDIASOURCES_MIN}")
plt.show()

## 8b. psfFlux − apFlux magnitude difference: dipole vs non-dipole

The **aperture correction** (difference between PSF-fit flux and fixed-aperture flux)
is a powerful discriminant between point sources and extended / artefact detections:

- For a **genuine point source** the PSF-fit flux and aperture flux should agree
  within noise → $\Delta m \approx 0$.
- For a **dipole artefact** the PSF fit is applied to a double-peaked residual image;
  the aperture captures both lobes while the PSF fit captures neither cleanly
  → $\Delta m$ can be systematically offset or show broader scatter.

We compute the AB-magnitude difference:

$$\Delta m = m_{\rm psf} - m_{\rm ap} = -2.5 \log_{10}\!\left(\frac{F_{\rm psf}}{F_{\rm ap}}\right)$$

and compare the distribution between dipole-flagged and non-dipole sources, per band.

In [ ]:
# ── Assemble one big DataFrame of all downloaded sources ─────────────────────
all_src_frames = []
for oid, df in src_cache.items():
    if df.empty:
        continue
    tmp = df.copy()
    tmp["diaObjectId_ext"] = oid
    tmp["field"] = presel[oid]["field"]
    all_src_frames.append(tmp)

if not all_src_frames:
    print("No diaSources available — skipping psfFlux-apFlux section.")
else:
    df_all_src = pd.concat(all_src_frames, ignore_index=True)
    df_all_src["is_dipole"] = df_all_src["r:isDipole"].fillna(False).astype(bool)

    # ── Compute Δm = m_psf − m_ap ─────────────────────────────────────────────
    has_ap = "r:apFlux" in df_all_src.columns and df_all_src["r:apFlux"].notna().any()
    if not has_ap:
        print("r:apFlux not available in downloaded sources — check that the API returns this column.")
    else:
        psf = pd.to_numeric(df_all_src["r:psfFlux"], errors="coerce")
        ap = pd.to_numeric(df_all_src["r:apFlux"], errors="coerce")
        with np.errstate(invalid="ignore", divide="ignore"):
            delta_m = np.where(
                (psf.values > 0) & (ap.values > 0),
                -2.5 * np.log10(psf.values / ap.values),
                np.nan,
            )
        df_all_src["delta_m_psf_ap"] = delta_m

        n_total = df_all_src["delta_m_psf_ap"].notna().sum()
        print(
            f"delta_m_psf_ap computed for {n_total:,} sources "
            f"({df_all_src['is_dipole'].sum():,} dipoles, "
            f"{(~df_all_src['is_dipole']).sum():,} non-dipoles)"
        )
        print(f"  overall median Δm = {np.nanmedian(delta_m):.4f} mag")
        print(f"  dipole  median Δm = {np.nanmedian(delta_m[df_all_src['is_dipole'].values]):.4f} mag")
        print(f"  non-dip median Δm = {np.nanmedian(delta_m[~df_all_src['is_dipole'].values]):.4f} mag")

In [ ]:
# ── Histograms Δm = psfFlux − apFlux : dipole vs non-dipole ──────────────────
if has_ap and "delta_m_psf_ap" in df_all_src.columns:
    # Clip range to ±1 mag for display (outliers are artefacts with very wrong flux)
    CLIP = 1.0
    BINS = np.linspace(-CLIP, CLIP, 81)

    # ── Global histogram (all bands combined) ─────────────────────────────────
    fig, ax = plt.subplots(figsize=(7, 4))

    dm_nd = df_all_src.loc[~df_all_src["is_dipole"], "delta_m_psf_ap"].dropna().values
    dm_dip = df_all_src.loc[df_all_src["is_dipole"], "delta_m_psf_ap"].dropna().values

    ax.hist(
        np.clip(dm_nd, -CLIP, CLIP),
        bins=BINS,
        density=True,
        alpha=0.55,
        color="steelblue",
        label=f"non-dipole  (N={len(dm_nd):,})",
    )
    ax.hist(
        np.clip(dm_dip, -CLIP, CLIP),
        bins=BINS,
        density=True,
        alpha=0.7,
        color="tomato",
        label=f"dipole  (N={len(dm_dip):,})",
    )

    ax.axvline(0, color="k", lw=1, ls="--", alpha=0.5)
    ax.axvline(
        float(np.nanmedian(dm_nd)),
        color="steelblue",
        lw=1.5,
        ls=":",
        label=f"non-dipole median = {np.nanmedian(dm_nd):.3f}",
    )
    ax.axvline(
        float(np.nanmedian(dm_dip)),
        color="tomato",
        lw=1.5,
        ls=":",
        label=f"dipole median = {np.nanmedian(dm_dip):.3f}",
    )

    ax.set_xlabel(r"$\Delta m = m_{\rm psf} - m_{\rm ap}$ (mag)")
    ax.set_ylabel("Probability density")
    ax.set_title(
        r"$m_{\rm psf} - m_{\rm ap}$ distribution — all bands combined" + "\n"
        f"Pre-selection: nDiaSrc >= {NDIASOURCES_MIN}"
    )
    ax.legend(fontsize=8)
    plt.tight_layout()
    savefig(f"delta_mag_psf_ap_all_bands_min{NDIASOURCES_MIN}")
    plt.show()

    # ── Per-band histograms ────────────────────────────────────────────────────
    if "r:band" in df_all_src.columns:
        bands_present = [b for b in BAND_ORDER if b in df_all_src["r:band"].values]
        ncols_b = min(3, len(bands_present))
        nrows_b = int(np.ceil(len(bands_present) / ncols_b))
        fig, axes = plt.subplots(nrows_b, ncols_b, figsize=(5 * ncols_b, 3.5 * nrows_b), squeeze=False)

        for bidx, band in enumerate(bands_present):
            ax = axes[bidx // ncols_b][bidx % ncols_b]
            sub = df_all_src[df_all_src["r:band"] == band]

            dm_nd_b = sub.loc[~sub["is_dipole"], "delta_m_psf_ap"].dropna().values
            dm_dip_b = sub.loc[sub["is_dipole"], "delta_m_psf_ap"].dropna().values

            color = BAND_COLORS[band]

            if len(dm_nd_b) > 0:
                ax.hist(
                    np.clip(dm_nd_b, -CLIP, CLIP),
                    bins=BINS,
                    density=True,
                    alpha=0.45,
                    color="steelblue",
                    label=f"non-dipole (N={len(dm_nd_b):,})",
                )
                ax.axvline(
                    float(np.nanmedian(dm_nd_b)),
                    color="steelblue",
                    lw=1.5,
                    ls=":",
                    label=f"med={np.nanmedian(dm_nd_b):.3f}",
                )

            if len(dm_dip_b) > 0:
                ax.hist(
                    np.clip(dm_dip_b, -CLIP, CLIP),
                    bins=BINS,
                    density=True,
                    alpha=0.7,
                    color=color,
                    label=f"dipole (N={len(dm_dip_b):,})",
                )
                ax.axvline(
                    float(np.nanmedian(dm_dip_b)),
                    color=color,
                    lw=1.5,
                    ls=":",
                    label=f"med={np.nanmedian(dm_dip_b):.3f}",
                )
            else:
                ax.text(0.5, 0.5, "no dipoles", ha="center", va="center", transform=ax.transAxes, fontsize=9)

            ax.axvline(0, color="k", lw=0.8, ls="--", alpha=0.4)
            ax.set_xlabel(r"$\Delta m$ (mag)")
            ax.set_ylabel("Density")
            ax.set_title(f"Band {band}", fontsize=9)
            ax.legend(fontsize=7)

        for bidx in range(len(bands_present), nrows_b * ncols_b):
            axes[bidx // ncols_b][bidx % ncols_b].set_visible(False)

        fig.suptitle(
            r"$m_{\rm psf} - m_{\rm ap}$ per band — dipole vs non-dipole" + f"\n[nDiaSrc>={NDIASOURCES_MIN}]",
            fontsize=11,
            y=1.01,
        )
        plt.tight_layout()
        savefig(f"delta_mag_psf_ap_per_band_min{NDIASOURCES_MIN}")
        plt.show()

## 9. Three-panel light curve for top high-dipole objects

**Panel 1 — psfFlux light curve**
- One colour per band.  Dipole detections circled with a grey open marker.
- Primary x-axis: MJD.  Secondary x-axis: date (YYYY-MM-DD).

**Panel 2 — nightly dipole histogram**
- Stacked bars per band; secondary y-axis: cumulative dipole count.

**Panel 3 — dipole morphology**
- `dipoleLength` (arcsec) — left y-axis.  `dipoleAngle` mod 360° — right y-axis.
- Stable angle → systematic template offset; scattered → noise artefact.

In [ ]:
def plot_object_lc(
    oid: str,
    df_src: pd.DataFrame,
    meta: dict,
    stat: dict,
    flux_col: str = "r:psfFlux",
    ferr_col: str = "r:psfFluxErr",
) -> None:
    """
    Three-panel light curve + nightly dipole histogram + dipole morphology.
    """
    if df_src.empty:
        print(f"  {oid}: empty diaSources — skipping.")
        return

    df = df_src.sort_values("r:midpointMjdTai").copy()
    df["is_dipole"] = df["r:isDipole"].fillna(False).astype(bool) if "r:isDipole" in df.columns else False
    mjd_all = pd.to_numeric(df["r:midpointMjdTai"], errors="coerce").values

    fig, axes = plt.subplots(
        3,
        1,
        figsize=(13, 10),
        gridspec_kw={"height_ratios": [3, 1.5, 1.5]},
    )

    # ── Panel 1 : psfFlux light curve ────────────────────────────────────────
    ax1 = axes[0]
    for band in BAND_ORDER:
        sub = df[df["r:band"] == band] if "r:band" in df.columns else pd.DataFrame()
        if sub.empty:
            continue
        flux = pd.to_numeric(sub[flux_col], errors="coerce").values
        ferr = pd.to_numeric(sub[ferr_col], errors="coerce").values if ferr_col in sub.columns else None
        mjd_b = pd.to_numeric(sub["r:midpointMjdTai"], errors="coerce").values
        color = BAND_COLORS[band]
        ax1.errorbar(
            mjd_b,
            flux,
            yerr=ferr,
            fmt="o",
            ms=4,
            lw=0.8,
            capsize=2,
            capthick=0.8,
            color=color,
            ecolor=color,
            alpha=0.8,
            label=f"{band} (n={len(sub)})",
        )
        dip = sub[sub["is_dipole"]]
        if not dip.empty:
            flux_d = pd.to_numeric(dip[flux_col], errors="coerce").values
            mjd_d = pd.to_numeric(dip["r:midpointMjdTai"], errors="coerce").values
            ax1.scatter(
                mjd_d,
                flux_d,
                s=130,
                facecolors="none",
                edgecolors="grey",
                linewidths=1.8,
                zorder=5,
                label=f"dipole {band} (n={len(dip)})",
            )

    ax1.axhline(0, color="k", lw=0.5, ls="--", alpha=0.4)
    ax1.set_ylabel(f"{flux_col.split(':')[1]} (nJy)")
    ax1.legend(loc="best", fontsize=7, ncol=4)
    title = (
        f"diaObjectId={oid}  field={meta['field']}  "
        f"nDiaSrc={meta['nDiaSources']}  downloaded={stat['n_src']}  "
        f"n_dip={stat['n_dipoles']}  frac={stat['dipole_fraction'] * 100:.1f}%"
    )
    if meta.get("gaia_name") and str(meta["gaia_name"]) not in ("nan", "None", ""):
        title += f"  Gaia={meta['gaia_name']}"
    if meta.get("simbad") and str(meta["simbad"]) not in ("nan", "None", ""):
        title += f"  SIMBAD={meta['simbad']}"
    ax1.set_title(title, fontsize=8)
    add_date_axis_on_top(ax1, mjd_all, n_ticks=8)

    # ── Panel 2 : nightly dipole histogram ───────────────────────────────────
    ax2 = axes[1]
    df_dip = df[df["is_dipole"]].copy()
    if not df_dip.empty and "r:band" in df_dip.columns:
        df_dip["night"] = np.floor(pd.to_numeric(df_dip["r:midpointMjdTai"], errors="coerce").values).astype(
            int
        )
        night_band = (
            df_dip.groupby(["night", "r:band"])
            .size()
            .unstack(fill_value=0)
            .reindex(columns=BAND_ORDER, fill_value=0)
        )
        night_band["total"] = night_band.sum(axis=1)
        nights_mjd = night_band.index.values.astype(float) + 0.5
        bottom = np.zeros(len(night_band))
        for band in BAND_ORDER:
            if band not in night_band.columns:
                continue
            vals = night_band[band].values.astype(float)
            ax2.bar(
                nights_mjd,
                vals,
                bottom=bottom,
                width=0.8,
                color=BAND_COLORS[band],
                edgecolor="white",
                lw=0.3,
                label=f"band {band}",
            )
            bottom += vals
        cum = np.cumsum(night_band["total"].values)
        ax2r = ax2.twinx()
        ax2r.step(nights_mjd, cum, where="post", color="k", lw=1.5, ls="--", alpha=0.6, label="cumulative")
        ax2r.set_ylabel("Cumulative dipoles", fontsize=8)
        ax2r.tick_params(axis="y", labelsize=8)
    ax2.set_ylabel("N dipoles per night")
    ax2.set_xlabel("MJD (TAI)")
    ax2.legend(loc="upper left", fontsize=7, ncol=3)

    finite_mjd = mjd_all[np.isfinite(mjd_all)]
    if len(finite_mjd) > 1:
        xlim = (finite_mjd.min() - 1, finite_mjd.max() + 1)
        ax1.set_xlim(xlim)
        ax2.set_xlim(xlim)

    # ── Panel 3 : dipole morphology ───────────────────────────────────────────
    ax3 = axes[2]
    if not df_dip.empty:
        for band in BAND_ORDER:
            sub = df_dip[df_dip["r:band"] == band] if "r:band" in df_dip.columns else pd.DataFrame()
            if sub.empty or "r:dipoleLength" not in sub.columns:
                continue
            dl = pd.to_numeric(sub["r:dipoleLength"], errors="coerce").values
            mjd_b = pd.to_numeric(sub["r:midpointMjdTai"], errors="coerce").values
            ax3.scatter(mjd_b, dl, s=25, color=BAND_COLORS[band], marker="o", label=f"length {band}")
        if "r:dipoleAngle" in df_dip.columns:
            ax3r = ax3.twinx()
            for band in BAND_ORDER:
                sub = df_dip[df_dip["r:band"] == band] if "r:band" in df_dip.columns else pd.DataFrame()
                if sub.empty:
                    continue
                da = pd.to_numeric(sub["r:dipoleAngle"], errors="coerce").values % 360
                mjd_b = pd.to_numeric(sub["r:midpointMjdTai"], errors="coerce").values
                ax3r.scatter(mjd_b, da, s=25, color=BAND_COLORS[band], marker="^", alpha=0.6)
            ax3r.set_ylabel("Dipole angle (deg)", fontsize=8, color="grey")
            ax3r.set_ylim(0, 360)
            ax3r.tick_params(axis="y", labelcolor="grey", labelsize=8)
        ax3.set_ylabel("Dipole length (arcsec)")
        ax3.set_xlabel("MJD (TAI)")
        ax3.legend(loc="upper left", fontsize=7, ncol=3)
        ax3.set_xlim(ax1.get_xlim())

    plt.tight_layout()
    savefig(f"lc_{oid}")
    plt.show()


print("plot_object_lc() defined.")

In [ ]:
for _, srow in top_sel.iterrows():
    oid = str(srow["diaObjectId"])
    print(f"\n=== {oid}  field={srow['field']}  nDiaSrc={srow['nDiaSources']}  n_dip={srow['n_dipoles']} ===")
    plot_object_lc(
        oid=oid,
        df_src=src_cache.get(oid, pd.DataFrame()),
        meta=presel[oid],
        stat=srow.to_dict(),
    )
print("Done.")

## 10. Angular stability of dipole direction per object

Circular standard deviation of the dipole position angle (folded mod 180°):

$$\sigma_{\theta} = \frac{1}{2}\sqrt{-2\ln\left|\langle e^{2i\theta}\rangle\right|}$$

- Small $\sigma_\theta$ → stable direction → systematic template mis-registration.
- Large $\sigma_\theta$ → random direction → noise artefact.

In [ ]:
def circular_std_deg(angles_deg: np.ndarray) -> float:
    """Circular std of angles (degrees), folded mod 180° (dipole symmetry)."""
    a = np.deg2rad(np.asarray(angles_deg, dtype=float) % 180)
    R = np.abs(np.mean(np.exp(2j * a)))
    return float(np.rad2deg(np.sqrt(-2 * np.log(R + 1e-12))) / 2)


angle_rows = []
for _, srow in top_sel.iterrows():
    oid = str(srow["diaObjectId"])
    df = src_cache.get(oid, pd.DataFrame())
    if df.empty or "r:isDipole" not in df.columns or "r:dipoleAngle" not in df.columns:
        continue
    df_dip = df[df["r:isDipole"].fillna(False).astype(bool)].copy()
    if df_dip.empty:
        continue
    angles = pd.to_numeric(df_dip["r:dipoleAngle"], errors="coerce").dropna().values
    if len(angles) < 2:
        continue
    row = {
        "diaObjectId": oid,
        "n_dipoles": srow["n_dipoles"],
        "field": srow["field"],
        "angle_mean_deg": float(
            np.rad2deg(np.angle(np.mean(np.exp(2j * np.deg2rad(angles % 180)))) / 2) % 180
        ),
        "angle_circ_std_deg": circular_std_deg(angles),
        "length_median_arcsec": float(
            pd.to_numeric(df_dip.get("r:dipoleLength", pd.Series(dtype=float)), errors="coerce").median()
        ),
    }
    if "r:band" in df_dip.columns:
        for band in BAND_ORDER:
            ang_b = (
                pd.to_numeric(
                    df_dip[df_dip["r:band"] == band].get("r:dipoleAngle", pd.Series(dtype=float)),
                    errors="coerce",
                )
                .dropna()
                .values
            )
            row[f"angle_cstd_{band}"] = circular_std_deg(ang_b) if len(ang_b) >= 2 else np.nan
    angle_rows.append(row)

df_angles = pd.DataFrame(angle_rows).sort_values("angle_circ_std_deg").reset_index(drop=True)
print("Angular stability of dipole direction (sorted by circular std):")
display(df_angles)
df_angles.to_csv(os.path.join(DIR_DATA, "dipole_angle_stability.csv"), index=False)

In [ ]:
# ── Rose histogram of dipole angles — top objects combined ────────────────────
all_angles, all_bands = [], []
for _, srow in top_sel.iterrows():
    oid = str(srow["diaObjectId"])
    df = src_cache.get(oid, pd.DataFrame())
    if df.empty or "r:isDipole" not in df.columns or "r:dipoleAngle" not in df.columns:
        continue
    df_dip = df[df["r:isDipole"].fillna(False).astype(bool)]
    ang = pd.to_numeric(df_dip["r:dipoleAngle"], errors="coerce").dropna().values
    bnd = df_dip["r:band"].values[: len(ang)] if "r:band" in df_dip.columns else ["?"] * len(ang)
    all_angles.extend(ang % 180)
    all_bands.extend(bnd)

if all_angles:
    n_bins = 36
    bins_a = np.linspace(0, 180, n_bins + 1)
    bottom = np.zeros(n_bins)
    fig, ax = plt.subplots(figsize=(7, 4))
    for band in BAND_ORDER:
        ang_b = np.array([a for a, b in zip(all_angles, all_bands) if b == band])
        if len(ang_b) == 0:
            continue
        cnt, _ = np.histogram(ang_b, bins=bins_a)
        ax.bar(
            (bins_a[:-1] + bins_a[1:]) / 2,
            cnt,
            bottom=bottom,
            width=180 / n_bins,
            color=BAND_COLORS[band],
            edgecolor="white",
            lw=0.3,
            label=f"band {band}",
        )
        bottom += cnt
    ax.set_xlabel("Dipole angle mod 180° (degrees)")
    ax.set_ylabel("N detections")
    ax.set_xlim(0, 180)
    ax.set_title(f"Dipole angle distribution — top {TOP_N_OBJECTS} objects\n(folded mod 180° for ±symmetry)")
    ax.legend(loc="upper right", fontsize=8, ncol=3)
    plt.tight_layout()
    savefig(f"dipole_angle_histogram_top{TOP_N_OBJECTS}")
    plt.show()
else:
    print("No dipole angles available.")

## 11. Per-field stacked histogram

In [ ]:
n_fields = len(DEEP_FIELDS)
ncols = min(3, n_fields)
nrows = int(np.ceil(n_fields / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows), squeeze=False)

for idx, fld in enumerate(DEEP_FIELDS):
    ax = axes[idx // ncols][idx % ncols]
    sub = df_stats[(df_stats["field"] == fld) & (df_stats["n_dipoles"] > 0)]
    if sub.empty:
        ax.set_title(f"{fld} — no dipoles")
        continue
    sub = sub.sort_values("n_dipoles", ascending=False)
    N_F = min(40, len(sub))
    top_f = sub.head(N_F)
    x_pos = np.arange(N_F)
    bottom = np.zeros(N_F)
    for band in BAND_ORDER:
        col = f"n_dip_{band}"
        if col not in top_f.columns:
            continue
        vals = top_f[col].values.astype(float)
        ax.bar(
            x_pos,
            vals,
            bottom=bottom,
            color=BAND_COLORS[band],
            edgecolor="white",
            lw=0.2,
            label=band,
            width=0.85,
        )
        bottom += vals
    ax.set_xticks(x_pos)
    ax.set_xticklabels([str(o) for o in top_f["diaObjectId"]], rotation=90, fontsize=5)
    ax.set_title(f"{fld}  ({len(sub)} with >0 dip, top {N_F})", fontsize=8)
    ax.set_ylabel("N dipoles")
    ax.legend(loc="upper right", fontsize=6, ncol=3)

for idx in range(n_fields, nrows * ncols):
    axes[idx // ncols][idx % ncols].set_visible(False)

fig.suptitle(f"Dipole count per diaObject — per DDF  [nDiaSrc>={NDIASOURCES_MIN}]", fontsize=11, y=1.01)
plt.tight_layout()
savefig(f"dipole_per_ddf_min{NDIASOURCES_MIN}")
plt.show()

## 12. Save all diaSources to Parquet

In [ ]:
subdir = os.path.join(DIR_DATA, "src_per_object")
os.makedirs(subdir, exist_ok=True)

if all_src_frames:
    df_all_src_save = (
        df_all_src.copy()
        if "df_all_src" in dir()
        else pd.concat(
            [
                {**df.copy(), "diaObjectId_ext": oid, "field": presel[oid]["field"]}
                for oid, df in src_cache.items()
                if not df.empty
            ],
            ignore_index=True,
        )
    )
    out_path = os.path.join(DIR_DATA, "all_src_presel.parquet")
    df_all_src_save.to_parquet(out_path, index=False)
    print(f"Saved {len(df_all_src_save):,} rows → {out_path}")

for oid, df in src_cache.items():
    if not df.empty:
        tmp = df.copy()
        tmp["field"] = presel[oid]["field"]
        tmp.to_parquet(os.path.join(subdir, f"{oid}_src.parquet"), index=False)

print(f"Per-object files in: {subdir}/")

## 13. Discussion and next steps

### Differences with `03_dipoleobjectcorr.ipynb`

| Step | `03` | `03b` (this notebook) |
|------|------|-----------------------|
| Alert catalogue | Cone-search API | Read parquet from `data_DIPOLES_01c/` |
| Crossmatch columns | Fetched on-the-fly | Already in the parquet |
| API calls | Cone-search + sources | **Sources only** |
| Consistency | Independent run | Fully consistent with `01c` and `02` |
| Lorenz curve | Yes | Yes + **Gini coefficient** |
| psfFlux − apFlux | No | **Yes** (section 8b) |

### Suggested follow-up notebooks

- **`04_dipole_cutouts.ipynb`**: fetch science/template/difference image triplets
  from Fink (`/api/v1/cutouts`) for the dipole-flagged visits.
- **`05_dipole_butler.ipynb`**: use `r:visit` + `r:detector` to locate the frames
  in the USDF Butler and check WCS registration quality.
- **`06_dipole_seeing.ipynb`**: join with `consDb` on `visitId` to correlate
  the per-visit dipole rate with seeing FWHM, airmass, and sky brightness.
